In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.ticker import MaxNLocator

from e_1_run_cvae import * 
from e_2_CVAE import *

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'van' # van or barr
model_type = 'bs' # hes or bs
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# training

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0
use_bn = False
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_NBN.pt"
resume_path = None # 이어서 학습하고 싶을 때


n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

학습 시작 | 이번 실행 chunks=5 | 진행 chunks=0->5 | files/epoch=100
Chunk step     1 | epoch    1 chunk   1/100 | file_idx  48 | Recon: 0.4213 | KL: 0.4085 | Total: 0.8298
Chunk step     2 | epoch    1 chunk   2/100 | file_idx   8 | Recon: 0.3576 | KL: 0.0068 | Total: 0.3644
Chunk step     3 | epoch    1 chunk   3/100 | file_idx  16 | Recon: 0.3698 | KL: 0.0007 | Total: 0.3705
Chunk step     4 | epoch    1 chunk   4/100 | file_idx  81 | Recon: 0.3728 | KL: 0.0005 | Total: 0.3733
Chunk step     5 | epoch    1 chunk   5/100 | file_idx   6 | Recon: 0.3630 | KL: 0.0003 | Total: 0.3633
모델 저장 완료: cvae_bs_van_8_128_4096_chunk5_NBN.pt
총 학습 시간: 13.11분 (0.22시간)
Training time: 788.803784s


In [ ]:
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_NBN.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_NBN.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: cvae_bs_van_8_128_4096_chunk5_NBN.pt | 완료 chunks=5
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=5->10 | files/epoch=100
Chunk step     6 | epoch    1 chunk   6/100 | file_idx  99 | Recon: 0.9767 | KL: 0.0000 | Total: 0.9768
Chunk step     7 | epoch    1 chunk   7/100 | file_idx  38 | Recon: 0.6162 | KL: 0.2412 | Total: 0.8573
Chunk step     8 | epoch    1 chunk   8/100 | file_idx   5 | Recon: 0.3736 | KL: 0.0039 | Total: 0.3775
Chunk step     9 | epoch    1 chunk   9/100 | file_idx  55 | Recon: 0.3664 | KL: 0.0003 | Total: 0.3667
Chunk step    10 | epoch    1 chunk  10/100 | file_idx  34 | Recon: 0.3678 | KL: 0.0002 | Total: 0.3680
모델 저장 완료: cvae_bs_van_8_128_4096_chunk10_NBN.pt
총 학습 시간: 15.43분 (0.26시간)
Training time: 928.078022s


In [36]:
num_chunks = 80
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk100_NBN.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk20_NBN.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

use_bn을 checkpoint 설정(False)으로 맞춥니다.
체크포인트 재개: cvae_bs_van_8_128_4096_chunk20_NBN.pt | 완료 chunks=20
학습 시작 | 이번 실행 chunks=80 | 진행 chunks=20->100 | files/epoch=100
Chunk step    21 | epoch    1 chunk  21/100 | file_idx  45 | Recon: 0.3681 | KL: 0.0013 | Total: 0.3694
Chunk step    22 | epoch    1 chunk  22/100 | file_idx  14 | Recon: 0.3684 | KL: 0.0006 | Total: 0.3689
Chunk step    23 | epoch    1 chunk  23/100 | file_idx  62 | Recon: 0.3720 | KL: 0.0003 | Total: 0.3723
Chunk step    24 | epoch    1 chunk  24/100 | file_idx  70 | Recon: 0.3677 | KL: 0.0002 | Total: 0.3679
Chunk step    25 | epoch    1 chunk  25/100 | file_idx  75 | Recon: 0.3696 | KL: 0.0002 | Total: 0.3698
Chunk step    26 | epoch    1 chunk  26/100 | file_idx  94 | Recon: 0.3695 | KL: 0.0002 | Total: 0.3697
Chunk step    27 | epoch    1 chunk  27/100 | file_idx  54 | Recon: 0.3734 | KL: 0.0002 | Total: 0.3736
Chunk step    28 | epoch    1 chunk  28/100 | file_idx  47 | Recon: 0.4063 | KL: 0.058

# use BN

In [21]:
use_bn = True
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_BN.pt"
resume_path = None

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

학습 시작 | 이번 실행 chunks=5 | 진행 chunks=0->5 | files/epoch=100
Chunk step     1 | epoch    1 chunk   1/100 | file_idx  48 | Recon: 0.4858 | KL: 0.3511 | Total: 0.8369
Chunk step     2 | epoch    1 chunk   2/100 | file_idx   8 | Recon: 0.3546 | KL: 0.0114 | Total: 0.3660
Chunk step     3 | epoch    1 chunk   3/100 | file_idx  16 | Recon: 0.3685 | KL: 0.0027 | Total: 0.3712
Chunk step     4 | epoch    1 chunk   4/100 | file_idx  81 | Recon: 0.3720 | KL: 0.0014 | Total: 0.3734
Chunk step     5 | epoch    1 chunk   5/100 | file_idx   6 | Recon: 0.3624 | KL: 0.0010 | Total: 0.3633
모델 저장 완료: cvae_bs_van_8_128_4096_chunk5_BN.pt
총 학습 시간: 13.90분 (0.23시간)
Training time: 836.757771s


In [23]:
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_BN.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_BN.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: cvae_bs_van_8_128_4096_chunk5_BN.pt | 완료 chunks=5
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=5->10 | files/epoch=100
Chunk step     6 | epoch    1 chunk   6/100 | file_idx  99 | Recon: 0.4684 | KL: 0.3574 | Total: 0.8258
Chunk step     7 | epoch    1 chunk   7/100 | file_idx  38 | Recon: 0.3852 | KL: 0.4349 | Total: 0.8201
Chunk step     8 | epoch    1 chunk   8/100 | file_idx   5 | Recon: 0.3615 | KL: 0.0095 | Total: 0.3710
Chunk step     9 | epoch    1 chunk   9/100 | file_idx  55 | Recon: 0.3560 | KL: 0.0047 | Total: 0.3608
Chunk step    10 | epoch    1 chunk  10/100 | file_idx  34 | Recon: 0.3577 | KL: 0.0042 | Total: 0.3619
모델 저장 완료: cvae_bs_van_8_128_4096_chunk10_BN.pt
총 학습 시간: 16.84분 (0.28시간)
Training time: 1012.629886s


In [25]:
num_chunks = 10
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk20_BN.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_BN.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    barr_type=barr_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    use_bn=use_bn,
    num_chunks=num_chunks,
    save_path=save_path,
    
    resume_path=resume_path,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: cvae_bs_van_8_128_4096_chunk10_BN.pt | 완료 chunks=10
학습 시작 | 이번 실행 chunks=10 | 진행 chunks=10->20 | files/epoch=100
Chunk step    11 | epoch    1 chunk  11/100 | file_idx  20 | Recon: 0.3617 | KL: 0.0035 | Total: 0.3652
Chunk step    12 | epoch    1 chunk  12/100 | file_idx  67 | Recon: 0.3926 | KL: 0.0645 | Total: 0.4571
Chunk step    13 | epoch    1 chunk  13/100 | file_idx  56 | Recon: 0.3621 | KL: 0.0035 | Total: 0.3657
Chunk step    14 | epoch    1 chunk  14/100 | file_idx  85 | Recon: 0.3667 | KL: 0.0019 | Total: 0.3687
Chunk step    15 | epoch    1 chunk  15/100 | file_idx  87 | Recon: 0.3840 | KL: 0.0652 | Total: 0.4492
Chunk step    16 | epoch    1 chunk  16/100 | file_idx  17 | Recon: 0.3841 | KL: 0.0665 | Total: 0.4506
Chunk step    17 | epoch    1 chunk  17/100 | file_idx  60 | Recon: 0.3617 | KL: 0.0032 | Total: 0.3649
Chunk step    18 | epoch    1 chunk  18/100 | file_idx  43 | Recon: 0.3572 | KL: 0.0014 | Total: 0.3586
Chunk step    19 | ep